<a href="https://colab.research.google.com/github/KasunUdayanga/NER-Sinhala-political-comment-identifier/blob/main/Notebooks/SinBERT_large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U transformers

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="NLPC-UOM/SinBERT-large")

config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("NLPC-UOM/SinBERT-large")
model = AutoModelForMaskedLM.from_pretrained("NLPC-UOM/SinBERT-large")

In [5]:
with open('/content/sinhala_dataset_balanced.conll', 'r') as f:
    for i in range(10):  # Read the first 10 lines
        print(f.readline(), end='')

පාලමුන B-LOC

ඔබතමයි B-Other
කාරයෝන්ට B-Other
රාජපක්ෂලාටම් B-PER
හර්ශ B-PER
විපක්ශයට B-ORG
අවලද B-Other
දැම්මාඇත්ත B-Other
ඩොබිලාට B-Other


# Task
Tokenize the dataset in the file "validation.conll" using a pre-trained model.

## Load and parse the dataset

### Subtask:
Read the CoNLL file and extract the tokens from each line, grouping them by sentence.


**Reasoning**:
Read the CoNLL file and extract tokens, grouping them by sentence as per the instructions.



In [6]:
sentences = []
labels = []
current_sentence = []
current_labels = []

with open('/content/sinhala_dataset_balanced.conll', 'r') as f:
    for line in f:
        stripped_line = line.strip()
        if not stripped_line:
            if current_sentence:
                sentences.append(" ".join(current_sentence))
                labels.append(current_labels)
                current_sentence = []
                current_labels = []
        else:
            token, label = stripped_line.split()
            current_sentence.append(token)
            current_labels.append(label)

if current_sentence:
    sentences.append(" ".join(current_sentence))
    labels.append(current_labels)

print("First 10 extracted sentences:")
print(sentences[:10]) # Print first 10 extracted sentences
print("\nFirst 10 extracted labels (for the first 10 sentences):")
print(labels[:10]) # Print first 10 extracted labels

First 10 extracted sentences:
['පාලමුන', 'ඔබතමයි කාරයෝන්ට රාජපක්ෂලාටම් හර්ශ විපක්ශයට අවලද දැම්මාඇත්ත ඩොබිලාට පැන්චො', 'මකුලෑව', 'නොදන්නා තනිම හෙනයක්පොලොන්නරුවෙ දෙයියන්ගෙම පිහිටයි ආයෙමත් පාර්ලිමේන්තුවට ගෙනාවොත් මදිවෙයි වැස්ස මිණීමරු අපාගත මුසිලයෝ බේරුනේ ශ්රිලනිපය අධිකරණයටම භාර ඉවරයිනෙ ඉතිහාසෙම කාපුපාවා', 'තරුණියන් දහස්ගනක් ටයර් සැයවල් අලුකලා හෑවා අබියොගයට ලක්කරන්න විරුද්ද ගෙනාවනම් මාමලාට ජවිප දෙබිඩි ගන්නවාට අමාත්යාංශය පාක්ෂිකයන් පුරවනවා බෝතලයක් වෙලානෙ නෑලුහැබැයි', 'නරකින් ජනාධිපතිතුමාටඔබට අප්පුහාමිලාට මකබ් බෑවිල පලයව ජෙප්පො තිරන ආන්ඩුවක අඩන්ගුවට', 'ඩිව්', 'දෙන්නැතිවෙයි ප්රසිද්ධියේ අප්රසිද්ධියේ කියලාජනාධිපතිවරණයත් සභාව මැතිවරණයත් පළාත් මාලි මාවට හාමුදුරුවෝ පීපිහිටයි සියයට හිටියදී විශ්වාසෙක පාර්ලිමිත්තයේ යුත්තෙක් බවයි නියාලුවන් කපටියන් අත්තනෝ', 'කෙහෙල්පන්නල', 'මීට වඩා ලැබෙයි මංත්ර්රීවරු අල්ලස් කලොත් කොමිෂමට පුලුවන්ද රැකපු ආඩම්බරකාර හයිය කණුව රැක්කා නෑජනතාව මොකහරි']

First 10 extracted labels (for the first 10 sentences):
[['B-LOC'], ['B-Other', 'B-Other', 'B-PER', 'B-PER', 'B-ORG', 'B-Oth

## Tokenize the dataset

### Subtask:
Use the loaded `AutoTokenizer` to tokenize the extracted sentences.


**Reasoning**:
Use the loaded tokenizer to tokenize the extracted sentences.



In [7]:
tokenized_sentences = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
print(tokenized_sentences.keys())
print(tokenized_sentences['input_ids'][:5])
print(tokenized_sentences['attention_mask'][:5])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


KeysView({'input_ids': tensor([[   0,  293,  268,  ...,    1,    1,    1],
        [   0,  663, 2140,  ...,    1,    1,    1],
        [   0,  361,  272,  ...,    1,    1,    1],
        ...,
        [   0,  267,  272,  ...,    1,    1,    1],
        [   0,  271,  276,  ...,    1,    1,    1],
        [   0,  308,  294,  ...,    1,    1,    1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})
tensor([[   0,  293,  268,  ...,    1,    1,    1],
        [   0,  663, 2140,  ...,    1,    1,    1],
        [   0,  361,  272,  ...,    1,    1,    1],
        [   0,  266,  296,  ...,    1,    1,    1],
        [   0,  336,  272,  ...,    1,    1,    1]])
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 

## Display tokenized output

### Subtask:
Show a sample of the tokenized data to verify the process.


**Reasoning**:
Decode a few tokenized sentences and print them along with the original sentences to verify the tokenization process.



In [8]:
# Decode and print a few tokenized sentences for verification
num_samples = 5
for i in range(num_samples):
    original_sentence = sentences[i]
    decoded_sentence = tokenizer.decode(tokenized_sentences['input_ids'][i], skip_special_tokens=True)
    print(f"Original: {original_sentence}")
    print(f"Tokenized (decoded): {decoded_sentence}")
    print("-" * 20)

Original: පාලමුන
Tokenized (decoded): පාලමුන
--------------------
Original: ඔබතමයි කාරයෝන්ට රාජපක්ෂලාටම් හර්ශ විපක්ශයට අවලද දැම්මාඇත්ත ඩොබිලාට පැන්චො
Tokenized (decoded): ඔබතමයි කාරයෝන්ට රාජපක්ෂලාටම් හර්ශ විපක්ශයට අවලද දැම්මාඇත්ත ඩොබිලාට පැන්චො
--------------------
Original: මකුලෑව
Tokenized (decoded): මකුලෑව
--------------------
Original: නොදන්නා තනිම හෙනයක්පොලොන්නරුවෙ දෙයියන්ගෙම පිහිටයි ආයෙමත් පාර්ලිමේන්තුවට ගෙනාවොත් මදිවෙයි වැස්ස මිණීමරු අපාගත මුසිලයෝ බේරුනේ ශ්රිලනිපය අධිකරණයටම භාර ඉවරයිනෙ ඉතිහාසෙම කාපුපාවා
Tokenized (decoded): නොදන්නා තනිම හෙනයක්පොලොන්නරුවෙ දෙයියන්ගෙම පිහිටයි ආයෙමත් පාර්ලිමේන්තුවට ගෙනාවොත් මදිවෙයි වැස්ස මිණීමරු අපාගත මුසිලයෝ බේරුනේ ශ්රිලනිපය අධිකරණයටම භාර ඉවරයිනෙ ඉතිහාසෙම කාපුපාවා
--------------------
Original: තරුණියන් දහස්ගනක් ටයර් සැයවල් අලුකලා හෑවා අබියොගයට ලක්කරන්න විරුද්ද ගෙනාවනම් මාමලාට ජවිප දෙබිඩි ගන්නවාට අමාත්යාංශය පාක්ෂිකයන් පුරවනවා බෝතලයක් වෙලානෙ නෑලුහැබැයි
Tokenized (decoded): තරුණියන් දහස්ගනක් ටයර් සැයවල් අලුකලා හෑවා අබියොගයට ලක්කරන්න විරුද්ද ගෙනාවනම් 

## Align Labels and Tokens

### Subtask:
Align the original labels with the tokenized input IDs.

**Reasoning**:
Align the original labels with the tokenized input IDs. This is necessary because tokenization can split words into sub-word units, and each sub-word unit needs to inherit the label of the original word.

In [9]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    # Handle cases where word_ids might be empty or None
    if not word_ids:
        return new_labels

    current_word_id = None
    for word_id in word_ids:
        if word_id is None:
            # Special token, assign a special label like -100
            new_labels.append(-100)
        elif word_id < len(labels):
            # Valid word_id within the range of original labels
            if word_id != current_word_id:
                # Start of a new word
                new_labels.append(labels[word_id])
                current_word_id = word_id
            else:
                # Continuation of a word, assign the same label as the first token of the word
                # We can access labels[word_id] again as word_id is the same as current_word_id
                 new_labels.append(labels[word_id])
        else:
            # word_id is out of bounds for the original labels list
            # This might happen with padding tokens beyond the original sentence length
            new_labels.append(-100) # Assign a special label for these tokens


    return new_labels

aligned_labels = []
for i in range(len(sentences)):
    word_ids = tokenized_sentences.word_ids(batch_index=i)
    aligned_labels.append(align_labels_with_tokens(labels[i], word_ids))

# Display a sample of aligned labels and corresponding tokens
num_samples_to_display = 5
for i in range(num_samples_to_display):
    print(f"Original Sentence: {sentences[i]}")
    print(f"Original Labels: {labels[i]}")
    tokenized_output = tokenizer.convert_ids_to_tokens(tokenized_sentences['input_ids'][i])
    print(f"Tokenized Words: {tokenized_output}")
    print(f"Aligned Labels: {aligned_labels[i]}")
    print("-" * 20)

Original Sentence: පාලමුන
Original Labels: ['B-LOC']
Tokenized Words: ['<s>', 'à¶´', 'à·ı', 'à¶½à¶¸', 'à·Ķ', 'à¶±', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<p

# Task
Prepare the custom dataset for model training by creating a label mapping, converting labels to numerical IDs, formatting the data as a Hugging Face dataset, and splitting it into training and evaluation sets.

## Create a label mapping

### Subtask:
Create a mapping from your string labels (e.g., "B-LOC", "O") to numerical IDs.


**Reasoning**:
Extract unique labels, create label-to-ID and ID-to-label mappings, and print them.



In [10]:
# 1. Extract all unique labels from the aligned_labels list, excluding -100
unique_labels = sorted(list(set(label for sentence_labels in aligned_labels for label in sentence_labels if label != -100)))

# 2. Create a dictionary that maps each unique string label to a unique integer ID
label_to_id = {label: i for i, label in enumerate(unique_labels)}

# 3. Create an inverse mapping (ID to label)
id_to_label = {i: label for label, i in label_to_id.items()}

# 4. Print the created label-to-ID and ID-to-label mappings
print("Label-to-ID mapping:")
print(label_to_id)
print("\nID-to-Label mapping:")
print(id_to_label)

Label-to-ID mapping:
{'B-LOC': 0, 'B-ORG': 1, 'B-Other': 2, 'B-PER': 3, 'I-LOC': 4, 'I-ORG': 5, 'I-Other': 6, 'I-PER': 7, 'O': 8}

ID-to-Label mapping:
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-Other', 3: 'B-PER', 4: 'I-LOC', 5: 'I-ORG', 6: 'I-Other', 7: 'I-PER', 8: 'O'}


## Convert aligned labels to ids

### Subtask:
Using the mapping, convert the aligned labels (which are currently strings) to numerical IDs.


**Reasoning**:
Convert the aligned string labels to numerical IDs using the created mapping, handling the special label -100.



In [11]:
numerical_aligned_labels = []
for sentence_labels in aligned_labels:
    numerical_sentence_labels = []
    for label in sentence_labels:
        if label == -100:
            numerical_sentence_labels.append(-100)
        else:
            numerical_sentence_labels.append(label_to_id[label])
    numerical_aligned_labels.append(numerical_sentence_labels)

# Print the first few lists of numerical labels to verify the conversion
print("First 5 lists of numerical aligned labels:")
for i in range(min(5, len(numerical_aligned_labels))):
    print(numerical_aligned_labels[i])

First 5 lists of numerical aligned labels:
[-100, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,

## Format the data as a hugging face dataset

### Subtask:
Organize the tokenized inputs (input IDs, attention masks) and the numerical label IDs into a Hugging Face Dataset object, which is a convenient format for training with the transformers library.


**Reasoning**:
Organize the tokenized inputs and numerical labels into a Hugging Face Dataset object.



In [12]:
from datasets import Dataset

data_dict = {
    'input_ids': tokenized_sentences['input_ids'],
    'attention_mask': tokenized_sentences['attention_mask'],
    'labels': numerical_aligned_labels
}

dataset = Dataset.from_dict(data_dict)
print(dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 9633
})


## Split the dataset

### Subtask:
Split the dataset into training and evaluation sets.


**Reasoning**:
Split the dataset into training and evaluation sets using the train_test_split method.



In [13]:
train_eval_dataset = dataset.train_test_split(test_size=0.2)
train_dataset = train_eval_dataset['train']
eval_dataset = train_eval_dataset['test']

print("Training dataset structure and size:")
print(train_dataset)

print("\nEvaluation dataset structure and size:")
print(eval_dataset)

Training dataset structure and size:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7706
})

Evaluation dataset structure and size:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1927
})


## Display Samples from Train and Evaluation Datasets

### Subtask:
Show sample entries from the training and evaluation datasets to inspect their structure and content.

**Reasoning**:
Display a few samples from the training and evaluation datasets to confirm that they contain the expected 'input_ids', 'attention_mask', and 'labels' and to get a sense of the data.

In [14]:
print("Sample from Training Dataset:")
train_sample = train_dataset[0]
print(f"Input IDs: {train_sample['input_ids'][:10]}") # Display first 10 input IDs
print(f"Decoded Text: {tokenizer.decode(train_sample['input_ids'], skip_special_tokens=True)[:100]}...") # Decode and display a portion of the text
print(f"Attention Mask: {train_sample['attention_mask'][:10]}") # Display first 10 attention mask values
print(f"Labels: {train_sample['labels'][:10]}") # Display first 10 labels

print("\nSample from Evaluation Dataset:")
eval_sample = eval_dataset[0]
print(f"Input IDs: {eval_sample['input_ids'][:10]}") # Display first 10 input IDs
print(f"Decoded Text: {tokenizer.decode(eval_sample['input_ids'], skip_special_tokens=True)[:100]}...") # Decode and display a portion of the text
print(f"Attention Mask: {eval_sample['attention_mask'][:10]}") # Display first 10 attention mask values
print(f"Labels: {eval_sample['labels'][:10]}") # Display first 10 labels

Sample from Training Dataset:
Input IDs: [0, 1784, 264, 302, 311, 293, 272, 270, 2, 1]
Decoded Text: අග්බෝපුර...
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Labels: [-100, 0, -100, -100, -100, -100, -100, -100, -100, -100]

Sample from Evaluation Dataset:
Input IDs: [0, 2190, 264, 273, 437, 268, 328, 355, 268, 275]
Decoded Text: පහත්ම අවසානය මහාසංඝ රත්නය හිගණ අපත විකෘතික දේමින් දේමල ධර්මයට චීවරධාරීන්ව උලුප්පමින් හරපද්දතියම පොලො...
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels: [-100, 2, 2, 2, 2, 2, 2, 2, 2, 2]


# Task
Train a Named Entity Recognition (NER) model using the custom dataset, evaluate its accuracy, and display the results.

## Load model for sequence labeling

### Subtask:
Load the pre-trained model for the token classification task.


**Reasoning**:
Import the necessary class and load the pre-trained model for token classification with the custom label mapping.



In [15]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "NLPC-UOM/SinBERT-large",
    num_labels=len(unique_labels),
    id2label=id_to_label,
    label2id=label_to_id
)

print(model.config.id2label)
print(model.config.label2id)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at NLPC-UOM/SinBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{0: 'B-LOC', 1: 'B-ORG', 2: 'B-Other', 3: 'B-PER', 4: 'I-LOC', 5: 'I-ORG', 6: 'I-Other', 7: 'I-PER', 8: 'O'}
{'B-LOC': 0, 'B-ORG': 1, 'B-Other': 2, 'B-PER': 3, 'I-LOC': 4, 'I-ORG': 5, 'I-Other': 6, 'I-PER': 7, 'O': 8}


## Define training arguments

### Subtask:
Specify the training parameters, such as the number of epochs, learning rate, and evaluation strategy.


**Reasoning**:
Import the TrainingArguments class and instantiate it with the specified parameters.



In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_model",  # Directory to save model checkpoints and outputs
    eval_strategy="epoch", # Evaluate every epoch
    learning_rate=2e-5,          # Learning rate
    per_device_train_batch_size=16, # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=10,           # Number of training epochs
    weight_decay=0.01,            # Weight decay
    push_to_hub=False,            # Do not push to Hugging Face Hub
)

print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,

## Define data collator

### Subtask:
Create a data collator to prepare batches of data for training.


**Reasoning**:
Import the necessary data collator and instantiate it with the tokenizer.



In [23]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print(data_collator)

DataCollatorForTokenClassification(tokenizer=RobertaTokenizerFast(name_or_path='NLPC-UOM/SinBERT-large', vocab_size=52000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	4: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
}
), padding=True, max_leng

## Define evaluation metrics

### Subtask:
Set up the metrics to evaluate the model's performance (e.g., accuracy, precision, recall, F1-score).


In [24]:
%pip install evaluate

In [25]:
%pip install seqeval

**Reasoning**:
Now that the `seqeval` library is installed, retry setting up the metrics by importing the necessary libraries, loading the seqeval metric, defining the `compute_metrics` function, and printing a confirmation message.



In [26]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (where label is -100)
    true_labels = [[id_to_label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

print("Metrics setup complete with compute_metrics function defined.")

Metrics setup complete with compute_metrics function defined.


## Train the model

### Subtask:
Train the model using the prepared dataset and defined training arguments.


**Reasoning**:
Import the Trainer class and instantiate it with the necessary components for training, then start the training process.



In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipython-input-2962370289.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kalihunter418 (kalihunter418-university-of-vauniya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.935701,0.729928,0.690309,0.709566,0.667793
2,1.062700,0.886778,0.755135,0.732865,0.743834,0.705481
3,0.822400,0.834549,0.782277,0.711139,0.745014,0.708918
4,0.670400,0.798853,0.794553,0.764596,0.779287,0.736240
5,0.565100,0.852607,0.798571,0.809243,0.803871,0.762718
6,0.479200,0.750413,0.809725,0.816858,0.813276,0.771619
7,0.420800,0.735971,0.816153,0.808048,0.812080,0.771337
8,0.377200,0.724426,0.820489,0.806033,0.813197,0.773534
9,0.347800,0.739975,0.812708,0.820293,0.816483,0.776576


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.935701,0.729928,0.690309,0.709566,0.667793
2,1.062700,0.886778,0.755135,0.732865,0.743834,0.705481
3,0.822400,0.834549,0.782277,0.711139,0.745014,0.708918
4,0.670400,0.798853,0.794553,0.764596,0.779287,0.736240
5,0.565100,0.852607,0.798571,0.809243,0.803871,0.762718
6,0.479200,0.750413,0.809725,0.816858,0.813276,0.771619
7,0.420800,0.735971,0.816153,0.808048,0.812080,0.771337
8,0.377200,0.724426,0.820489,0.806033,0.813197,0.773534
9,0.347800,0.739975,0.812708,0.820293,0.816483,0.776576
10,0.343400,0.755614,0.813285,0.826340,0.819761,0.780125


TrainOutput(global_step=4820, training_loss=0.5490915781234804, metrics={'train_runtime': 5966.8986, 'train_samples_per_second': 12.915, 'train_steps_per_second': 0.808, 'total_flos': 1.400137040666736e+16, 'train_loss': 0.5490915781234804, 'epoch': 10.0})

## Save the trained model

### Subtask:
Save the trained model to a directory for later use or download.

**Reasoning**:
Save the trained model using the `save_model` method of the Trainer object.

In [ ]:
# Save the trained model
trainer.save_model("./ner_model_saved")
print("Model saved successfully to ./ner_model_saved")